# ecoli1: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('11_ecoli1_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,86.415,81.710,87.073,78.083,96.116,87.100,89.468,95.473,84.063
smote,89.765,80.711,74.380,90.375,89.736,90.056,87.700,95.806,85.238
bsmote,89.748,80.528,73.712,90.375,89.732,90.054,87.003,95.763,85.138
ros,88.308,78.745,72.504,88.375,88.763,88.569,87.284,95.611,83.526
adasyn,88.646,78.308,70.402,91.000,87.021,89.011,85.771,95.373,83.477
casb,88.341,78.619,72.051,88.375,88.771,88.573,87.903,95.839,83.480
casb_relaxed,88.560,79.174,72.933,88.417,89.155,88.786,88.151,95.833,83.867
casb_edited,88.686,76.701,66.045,92.375,85.494,88.934,85.609,95.331,82.693
casb_ros,88.563,78.739,71.795,89.042,88.575,88.808,88.151,95.899,83.651


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,84.902,79.933,86.288,75.250,96.124,85.687,87.024,95.537,82.417
smote,88.307,79.486,74.574,86.417,90.713,88.565,86.383,95.383,83.896
bsmote,86.001,75.477,69.101,85.000,87.606,86.303,81.540,94.443,80.739
ros,87.352,78.031,72.857,85.000,90.147,87.574,84.550,94.982,82.691
adasyn,87.604,77.069,69.754,87.667,88.009,87.838,84.355,94.979,82.336
casb,87.277,77.507,71.659,85.667,89.367,87.517,86.253,95.421,82.392
casb_relaxed,87.707,78.663,73.581,85.667,90.321,87.994,84.728,95.102,83.185
casb_edited,89.169,78.708,70.213,91.000,87.805,89.403,79.039,94.282,83.938
casb_ros,86.913,77.253,72.000,85.000,89.559,87.279,85.303,95.166,82.083


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,85.881,78.958,79.631,79.208,93.413,86.310,89.531,96.066,82.420
smote,89.730,81.317,75.643,88.958,90.897,89.928,88.372,96.074,85.524
bsmote,89.877,80.284,72.635,90.958,89.163,90.061,86.993,95.661,85.081
ros,89.251,80.970,76.093,87.625,91.290,89.457,89.526,96.133,85.110
adasyn,90.559,80.871,72.365,92.958,88.571,90.765,87.289,95.955,85.715
casb,90.678,82.210,75.748,90.917,90.709,90.813,89.570,96.157,86.444
casb_relaxed,89.404,80.584,74.709,88.917,90.321,89.619,89.564,96.308,84.994
casb_edited,89.804,79.458,70.670,92.250,87.805,90.028,83.197,95.181,84.631
casb_ros,89.816,80.930,74.380,89.667,90.324,89.995,89.980,96.273,85.373


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,82.710,75.925,80.048,72.917,94.186,83.551,88.237,95.486,79.317
smote,85.911,75.817,70.375,83.958,88.371,86.165,87.624,95.390,80.864
bsmote,86.390,75.794,69.220,86.500,87.206,86.853,88.137,95.622,81.092
ros,84.790,74.740,70.181,81.375,88.759,85.067,88.018,95.491,79.765
adasyn,86.943,75.711,67.565,88.375,86.244,87.310,87.978,95.567,81.327
casb,85.957,75.793,70.159,84.542,87.979,86.260,88.145,95.539,80.875
casb_relaxed,85.749,75.539,69.755,83.917,87.979,85.948,87.869,95.464,80.644
casb_edited,86.758,75.262,66.726,88.500,85.664,87.082,86.938,95.138,81.010
casb_ros,86.564,76.445,70.505,85.833,87.979,86.906,88.068,95.478,81.505


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,84.977,79.131,83.260,76.365,94.959,85.662,88.565,95.640,82.054
smote,88.428,79.333,73.743,87.427,89.929,88.678,87.520,95.663,83.881
bsmote,88.004,78.021,71.167,88.208,88.427,88.317,85.918,95.372,83.013
ros,87.425,78.121,72.909,85.594,89.740,87.667,87.344,95.554,82.773
adasyn,88.438,77.990,70.022,90.000,87.461,88.731,86.348,95.469,83.214
casb,88.063,78.532,72.404,87.375,89.206,88.291,87.968,95.739,83.298
casb_relaxed,87.855,78.490,72.744,86.729,89.444,88.086,87.578,95.677,83.172
casb_edited,88.604,77.532,68.413,91.031,86.692,88.862,83.696,94.983,83.068
casb_ros,87.964,78.342,72.170,87.385,89.109,88.247,87.876,95.704,83.153


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,88.686,smote,89.765,-1.080,LOSS
1,f1,casb_relaxed,79.174,smote,80.711,-1.537,LOSS
2,precision,casb_relaxed,72.933,smote,74.380,-1.447,LOSS
3,recall,casb_edited,92.375,adasyn,91.000,1.375,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,89.169,smote,88.307,0.862,WIN
1,f1,casb_edited,78.708,smote,79.486,-0.778,LOSS
2,precision,casb_relaxed,73.581,smote,74.574,-0.993,LOSS
3,recall,casb_edited,91.000,adasyn,87.667,3.333,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,90.678,adasyn,90.559,0.119,WIN
1,f1,casb,82.210,smote,81.317,0.893,WIN
2,precision,casb,75.748,ros,76.093,-0.345,LOSS
3,recall,casb_edited,92.250,adasyn,92.958,-0.708,LOSS


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,88.604,adasyn,88.438,0.166,WIN
1,f1,casb,78.532,smote,79.333,-0.800,LOSS
2,precision,casb_relaxed,72.744,smote,73.743,-0.999,LOSS
3,recall,casb_edited,91.031,adasyn,90.000,1.031,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.775217,0.714286,0.833333,0.625000,0.961538,0.793269,0.778602,0.911058,0.744751,None
1,42,1,svm,smote,0.50,0.840673,0.774194,0.800000,0.750000,0.942308,0.846154,0.776265,0.920673,0.807433,None
2,42,1,svm,bsmote,0.50,0.832050,0.750000,0.750000,0.750000,0.923077,0.836538,0.757451,0.923077,0.791025,None
3,42,1,svm,ros,0.50,0.832050,0.750000,0.750000,0.750000,0.923077,0.836538,0.742646,0.906250,0.791025,None
4,42,1,svm,adasyn,0.50,0.823337,0.727273,0.705882,0.750000,0.903846,0.826923,0.758499,0.920673,0.775305,None
5,42,1,svm,casb,1.00,0.814531,0.705882,0.666667,0.750000,0.884615,0.817308,0.760988,0.924279,0.760207,None
6,42,1,svm,casb_relaxed,0.50,0.840673,0.774194,0.800000,0.750000,0.942308,0.846154,0.795665,0.923077,0.807433,None
7,42,1,svm,casb_edited,1.00,0.805629,0.685714,0.631579,0.750000,0.865385,0.807692,0.772051,0.930288,0.745672,None
8,42,1,svm,casb_ros,0.50,0.832050,0.750000,0.750000,0.750000,0.923077,0.836538,0.790729,0.931490,0.791025,None
9,42,1,knn,original,0.50,0.813055,0.758621,0.846154,0.687500,0.961538,0.824519,0.727550,0.892428,0.785838,None
